# Análisis de rentabilidad de una empresa comercial

**Introducción**

El objetivo de este análisis es evaluar el desempeño comercial del año 2024 de una empresa que vende artículos deportivos, como relojes, sensores, bandas, etc. Para encontrar debilidades y fortalezas, y posteriormente realizar recomendaciones accionables basados en los insights principales.
Se trabajará con los siguientes datasets:
- VENTAS EIG 2024.xlsx → Información detallada de facturas emitidas, productos, cantidades, centros de costo, ventas totales.
- costos_2024.xlsx → Información detallada de nombres de productos con sus respectivos costos de compra.

El análisis sigue una lógica clara y progresiva:

🔍 Evaluar la confiabilidad de los datos (calidad de datos en Python)

💰 Analizar si el negocio es rentable (revenue, costos y ganancia)

📊 Comunicar los resultados (Resumen ejecutivo)

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer recomendaciones accionables:

- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿El negocio es rentable? (calcular ganancia o profit)
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Por cuál centro de costo se vendió más?
- ¿Cuáles centros de costos tienen el mayor margen de utilidad?

## Cargar y validar la calidad de los datos

In [2]:
# Importar librerías
import pandas as pd

In [3]:
# Cargar dataset
ventas= pd.read_excel('VENTAS EIG 2024.xlsx')

In [4]:
# Explorar dataset
print("Información del dataset")
ventas.info()
print()
print('Vista previa del dataset')
ventas.head()

Información del dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4577 entries, 0 to 4576
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Código contable     4577 non-null   int64         
 1   Cuenta contable     4577 non-null   object        
 2   Comprobante         4577 non-null   object        
 3   Fecha registro      4577 non-null   datetime64[ns]
 4   Fecha venta         4577 non-null   datetime64[ns]
 5   Identificación      4577 non-null   int64         
 6   Nombre del tercero  4577 non-null   object        
 7   Producto            4577 non-null   object        
 8   Detalle             4577 non-null   object        
 9   Cantidad            4577 non-null   int64         
 10  Centro de costo     4577 non-null   object        
 11  Ventas              4577 non-null   float64       
dtypes: datetime64[ns](2), float64(1), int64(3), object(6)
memory usage: 429.2+ KB

V

,Código contable,Cuenta contable,Comprobante,Fecha registro,Fecha venta,Identificación,Nombre del tercero,Producto,Detalle,Cantidad,Centro de costo,Ventas
0,41353001,Comercio al por mayor y al detal,FV-2-837,2024-01-01,2024-01-01,6221099,JULIAN VIERA ZORRILLA,Mantenimiento 1,Producto: Mantenimiento 1 Cant: 1.00,1,VENTAS SERVICIO TÉCNICO,29411.76
1,41353001,Comercio al por mayor y al detal,FV-2-837,2024-01-01,2024-01-01,6221099,JULIAN VIERA ZORRILLA,CABLE VANTAGE/IGNITE/GRITX,Producto: CABLE VANTAGE/IGNITE/GRITX Bod: 4 Ca...,1,VENTAS SERVICIO TÉCNICO,110924.37
2,41353001,Comercio al por mayor y al detal,FV-3-769,2024-01-02,2024-01-02,1030559233,JOHN WILLIAM ACERO BRAUSIN,WRIST STRAP SET VANTAGE M WHI-M,Producto: WRIST STRAP SET VANTAGE M WHI-M Bod:...,1,VENTAS TIENDA 127,113445.38
3,41353001,Comercio al por mayor y al detal,FV-3-770,2024-01-02,2024-01-02,80216636,ARIEL LEANDRO ALVARADO AGUILLON,Mantenimiento 2,Producto: Mantenimiento 2 Cant: 1.00,1,VENTAS TIENDA 127,50420.17
4,41353001,Comercio al por mayor y al detal,FV-3-771,2024-01-02,2024-01-02,80087680,SANTIAGO VALLEJO CANO,POLAR PRO CHEST STRAP BLK M-XXL GEN,Producto: POLAR PRO CHEST STRAP BLK M-XXL GEN ...,1,VENTAS TIENDA 127,180672.27


El dataset de ventas tiene 4.577 registros no nulos, los formatos de los datos están acordes para un correcto análisis.

In [5]:
# Revisar el rango de fechas para detectar años fuera de rango
print(f"Fecha registro\n{ventas['Fecha registro'].dt.year.value_counts()}")
print(f"\nFecha venta\n{ventas['Fecha venta'].dt.year.value_counts()}")

Fecha registro
2024    4577
Name: Fecha registro, dtype: int64

Fecha venta
2024    4577
Name: Fecha venta, dtype: int64


Correcto, todas las fechas están dentro del año 2024, el cual es objeto de análisis.

In [6]:
# Renombrar columna para evitar confusión con nombre del dataset
ventas= ventas.rename(columns={'Ventas ': 'Venta total'})

In [7]:
# Revisar valores nulos para columnas númericas
ventas[['Cantidad', 'Venta total', 'Identificación']].isna().sum()

Cantidad          0
Venta total       0
Identificación    0
dtype: int64

In [8]:
# Revisar variables numéricas
# Revisar variables con errores

# Revisar cantidad
print("Cantidad <= 0:")
print(ventas[ventas['Cantidad'] <= 0]['Cantidad'].value_counts())

Cantidad <= 0:
Series([], Name: Cantidad, dtype: int64)


In [9]:
# Revisar Venta total
print("Venta total <= 0:")
print(ventas[ventas['Venta total'] <= 0]['Venta total'].value_counts())

Venta total <= 0:
0.0    172
Name: Venta total, dtype: int64


Hay 172 registros con venta total =0 
Estos registros corresponden a obsequios a clientes, se eliminarán para no sesgar el resultado del análisis.

In [10]:
# Eliminar registros con ventas =0
print("Registros antes:", len(ventas))
ventas_clean = ventas[ventas['Venta total'] != 0]
print("Registros después:", len(ventas_clean))
print("Se eliminaron 172 registros con venta total =0")

Registros antes: 4577
Registros después: 4405
Se eliminaron 172 registros con venta total =0


In [11]:
# Verificar cambios
ventas_clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4405 entries, 0 to 4576
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Código contable     4405 non-null   int64         
 1   Cuenta contable     4405 non-null   object        
 2   Comprobante         4405 non-null   object        
 3   Fecha registro      4405 non-null   datetime64[ns]
 4   Fecha venta         4405 non-null   datetime64[ns]
 5   Identificación      4405 non-null   int64         
 6   Nombre del tercero  4405 non-null   object        
 7   Producto            4405 non-null   object        
 8   Detalle             4405 non-null   object        
 9   Cantidad            4405 non-null   int64         
 10  Centro de costo     4405 non-null   object        
 11  Venta total         4405 non-null   float64       
dtypes: datetime64[ns](2), float64(1), int64(3), object(6)
memory usage: 447.4+ KB


In [12]:
# Revisar nombre de columnas
ventas_clean.columns

Index(['Código contable', 'Cuenta contable', 'Comprobante', 'Fecha registro',
       'Fecha venta', 'Identificación', 'Nombre del tercero', 'Producto',
       'Detalle', 'Cantidad', 'Centro de costo', 'Venta total'],
      dtype='object')

In [13]:
# Estandarizar datos
# Contar valores únicos
ventas_clean['Centro de costo'].value_counts()

VENTAS TIENDA 127          2360
VENTAS PÁGINA WEB           911
VENTAS REDES SOCIALES       563
VENTAS DISTRIBUCIÓN         330
VENTAS SERVICIO TÉCNICO     163
PREVENTAS                    72
EVENTOS                       4
ADMINISTRACION                2
Name: Centro de costo, dtype: int64

Los nombres de la variable `Centro de costo` están estandarizados.

In [14]:
# Revisar y eliminar registros duplicados
duplicados_ventas_clean = ventas_clean.duplicated()
print(f"Registros duplicados ventas_clean: {duplicados_ventas_clean.sum()}")

Registros duplicados ventas_clean: 2


In [15]:
# Consultar y visualizar filas completamente duplicadas
duplicados = ventas_clean[ventas_clean.duplicated(keep=False)]
display(duplicados.T)

,438,439,577,578
Código contable,41353001,41353001,41353001,41353001
Cuenta contable,Comercio al por mayor y al detal,Comercio al por mayor y al detal,Comercio al por mayor y al detal,Comercio al por mayor y al detal
Comprobante,FV-2-900,FV-2-900,FV-2-922,FV-2-922
Fecha registro,2024-02-07 00:00:00,2024-02-07 00:00:00,2024-02-13 00:00:00,2024-02-13 00:00:00
Fecha venta,2024-02-07 00:00:00,2024-02-07 00:00:00,2024-02-13 00:00:00,2024-02-13 00:00:00
Identificación,93413783,93413783,1019098881,1019098881
Nombre del tercero,JERSON ANDRES GALINDO,JERSON ANDRES GALINDO,JOYERIA Y RELOJERIA MOISES ALVARADO JR,JOYERIA Y RELOJERIA MOISES ALVARADO JR
Producto,POLAR PACER BLK/BLK S-L,POLAR PACER BLK/BLK S-L,POLAR PACER PUR/PUR S-L,POLAR PACER PUR/PUR S-L
Detalle,Producto: POLAR PACER BLK/BLK S-L Bod: 1 Cant:...,Producto: POLAR PACER BLK/BLK S-L Bod: 1 Cant:...,Producto: POLAR PACER PUR/PUR S-L Bod: 1 Cant:...,Producto: POLAR PACER PUR/PUR S-L Bod: 1 Cant:...
Cantidad,1,1,1,1


In [16]:
# Eliminar registros duplicados exactos
ventas_clean = ventas_clean.drop_duplicates()

In [17]:
# Verificar cambios
ventas_clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4403 entries, 0 to 4576
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Código contable     4403 non-null   int64         
 1   Cuenta contable     4403 non-null   object        
 2   Comprobante         4403 non-null   object        
 3   Fecha registro      4403 non-null   datetime64[ns]
 4   Fecha venta         4403 non-null   datetime64[ns]
 5   Identificación      4403 non-null   int64         
 6   Nombre del tercero  4403 non-null   object        
 7   Producto            4403 non-null   object        
 8   Detalle             4403 non-null   object        
 9   Cantidad            4403 non-null   int64         
 10  Centro de costo     4403 non-null   object        
 11  Venta total         4403 non-null   float64       
dtypes: datetime64[ns](2), float64(1), int64(3), object(6)
memory usage: 447.2+ KB


Se visualizan 2 registros menos, que corresponden a los duplicados.

## Analizar rentabilidad del negocio

### Cálculo de KPIs principales

🎯 Objetivo: Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usarán los datasets (ventas_clean y costos_2024):

📊 Parte 1: Rentabilidad del negocio

- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿El negocio es rentable? (calcular ganancia o profit)

In [18]:
# Rentabilidad del negocio
# Calcular ingreso total (revenue)
ingreso_total= ventas_clean['Venta total'].sum()
print(f'Ingreso total: {ingreso_total:,.2f}')

Ingreso total: 2,719,633,670.59


In [19]:
# Calcular costo total de productos
# cargar dataset de costos
costos = pd.read_excel('costos_2024.xlsx')

In [20]:
# Explorar dataset de costos
costos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Producto    277 non-null    object 
 1   Costo       277 non-null    float64
 2   Unnamed: 2  0 non-null      float64
 3   Unnamed: 3  0 non-null      float64
dtypes: float64(3), object(1)
memory usage: 8.8+ KB


Se visualizan 277 registros no nulos en el dataset costos

In [21]:
# Modificar valores costo a 2 decimales
costos['Costo'] = costos['Costo'].round(2)

# Eliminar columnas sin datos
costos = costos.dropna(axis=1, how='all')

# Verificar cambios
costos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Producto  277 non-null    object 
 1   Costo     277 non-null    float64
dtypes: float64(1), object(1)
memory usage: 4.5+ KB


In [22]:
# Unir ventas_clean con costos para unificar dataset
ventas_clean = ventas_clean.merge(
    costos[['Producto', 'Costo']],
    on= 'Producto', 
    how = 'left'
)

In [23]:
# Calcular el costo total de cada venta
ventas_clean['costo_total_pedido'] = ventas_clean['Cantidad'] * ventas_clean['Costo']

In [24]:
# Sumar costo total de pedidos
costo_general = ventas_clean['costo_total_pedido'].sum()
print(f"Costo general: {costo_general:,.2f}")

Costo general: 1,522,922,457.52


In [25]:
# Calcular ganancia total
ganancia = ingreso_total - costo_general
print(f"Ganancia total: {ganancia:,.2f}")

Ganancia total: 1,196,711,213.07


In [26]:
# Calcular margen de utilidad (porcentaje)
margen_utilidad = (ganancia / ingreso_total) *100
print(f"Margen de utilidad: {margen_utilidad:.2f}%")

Margen de utilidad: 44.00%


El negocio es muy rentable, genera una utilidad bruta de 1.196.711.213 con un margen de utilidad del 44%

In [27]:
# Crear una columna para ganancia de cada pedido
ventas_clean['ganancia_pedido'] = ventas_clean['Venta total'] - ventas_clean['costo_total_pedido']

In [28]:
# Crear una columna para margen de utilidad de cada pedido
ventas_clean['margen_utilidad'] = (ventas_clean['ganancia_pedido'] / ventas_clean['Venta total'] * 100).round(2)

📈 Parte 2: Comportamiento de ventas

- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Por cuál centro de costo se vendió más?
- ¿Cuáles centros de costos tienen el mayor margen de utilidad?

In [29]:
# Calcular cantidad promedio de productos por orden
cantidad_promedio_productos = ventas_clean['Cantidad'].sum() / ventas_clean['Comprobante'].nunique()
print(f'Cantidad promedio de productos por orden: {cantidad_promedio_productos:.0f}')

Cantidad promedio de productos por orden: 2


In [30]:
# Calcular ticket promedio
ticket_promedio = ventas_clean['Venta total'].sum() / ventas_clean['Comprobante'].nunique()
print(f'Ticket promedio: {ticket_promedio:,.0f}')

Ticket promedio: 824,881


In [31]:
# Visualizar los productos más vendidos
# Sumar la cantidad total vendida de cada producto
producto_mas_vendido_cantidad = ventas_clean.groupby('Producto')['Cantidad'].sum().sort_values(ascending=False)
print("Ranking de productos más vendidos:")
print(producto_mas_vendido_cantidad.head(6))

Ranking de productos más vendidos:
Producto
POLAR H9 HR SENSOR BLE BLK M-XXL               403
POLAR UNITE BLK S-L T                          387
POLAR PACER BLK/BLK S-L                        384
PILA LITHIUM GP CR 2025 3 V 160 MAH BLISTER    372
POLAR OH1 N OHR SENSOR BLK                     353
POLAR H10 N HR SENSOR BLK M                    342
Name: Cantidad, dtype: int64


El producto más vendido es la POLAR H9 HR SENSOR con 403 unidades totales.

In [32]:
# Ventas por centros de costo
ventas_centro_costos = ventas_clean.groupby('Centro de costo')['Venta total'].sum().sort_values(ascending=False).astype(int)
print("Ranking por Centro de Costos:")
print(ventas_centro_costos.head(8))

Ranking por Centro de Costos:
Centro de costo
VENTAS TIENDA 127          851359857
VENTAS DISTRIBUCIÓN        791996747
VENTAS PÁGINA WEB          604833522
VENTAS REDES SOCIALES      340928873
VENTAS SERVICIO TÉCNICO     65779096
PREVENTAS                   58244627
EVENTOS                      6213445
ADMINISTRACION                277500
Name: Venta total, dtype: int64


El centro de costos con mayores ingresos fue Ventas tienda 127 con 851.359.857

In [38]:
# Márgenes por centros de costo
margen_centro_costos = ventas_clean.groupby('Centro de costo')['margen_utilidad'].mean().round(2).sort_values(ascending=False)
print("Mejores Márgenes por Centro de Costos:")
print(margen_centro_costos.head(5))

Mejores Márgenes por Centro de Costos:
Centro de costo
VENTAS PÁGINA WEB          39.53
VENTAS REDES SOCIALES      38.45
PREVENTAS                  37.44
VENTAS DISTRIBUCIÓN        32.36
VENTAS SERVICIO TÉCNICO     9.32
Name: margen_utilidad, dtype: float64


El centro de costo VENTAS PÁGINA WEB tiene el mayor margen de utilidad con 39.53%

## Resumen ejecutivo

**Hallazgos clave**

- Se encontaron 172 registros con venta total =0 Estos registros corresponden a obsequios a clientes, se eliminaron para no sesgar el resultado del análisis.
- Se encontaron 2 registros exactos duplicados en el dataset ventas, los registros 439 y 578 los cuales se eliminaron para optimizar el análisis.

**Insights accionables**

- El Ingreso total del año 2024 fue 2.719.633.671
- Costo general: 1.522.922.458
- Ganancia total: 1.196.711.213
- El negocio es muy rentable, genera una utilidad bruta de 1.196.711.213 con un margen de utilidad del 44%
- Cantidad promedio de productos por orden: 2
- Ticket promedio: 824.881
- El producto más vendido es la POLAR H9 HR SENSOR con 403 unidades totales.
- El centro de costos con mayores ingresos fue VENTAS TIENDA 127 con 851.359.857
- El centro de costo VENTAS PÁGINA WEB tiene el mayor margen de utilidad con 39.53% seguido de VENTAS REDES SOCIALES con 38.45%

**Recomendaciones**

El top de ventas está dominado por sensores de frecuencia cardíaca y relojes deportivos de la marca Polar (HR Sensor, Unite, Pacer, OH1, Verity Sense), lo que sugiere que el negocio está enfocado en tecnología wearable para deporte. Esto es valioso para tomar decisiones de inventario. 
- Conviene asegurar stock prioritario de estos modelos y negociar mejores condiciones con el proveedor dado el volumen de ventas que representan.
- La "PILA LITHIUM GP CR 2025" tiene altísima rotación (372 unidades). Esto es una oportunidad de ingreso recurrente: se podrían ofrecer packs de baterías de repuesto o recordatorios de recompra a clientes que ya compraron sensores, generando ventas repetidas con bajo esfuerzo comercial.
- El promedio de productos por orden es de solo 2, con un ticket promedio de 824.881. Hay una oportunidad clara de venta cruzada: armar combos (por ejemplo, sensor + correa + batería, o reloj + accesorios) para subir el tamaño del carrito.
- Tienda 127 y Distribución concentran cerca del 60% de los ingresos totales, mientras que Página Web y Redes Sociales, aunque menores, ya representan una porción relevante (en conjunto cerca de 35%). Dado que estos canales digitales suelen tener menor costo operativo que el retail físico, se debe evaluar posible aumento de inversión en marketing.
- Valdría la pena estudiar qué prácticas hacen destacar a la Tienda 127 (ubicación, vendedores, mix de productos, promociones) para replicarlas.
- Servicio Técnico, Preventas, Eventos y Administración representan una fracción mínima de los ingresos. Esto no necesariamente es un problema (no todos los canales deben generar ingresos altos). Se debe evaluar estrategias de incremento de ventas como gasto de marketing, contacto clientes sin ventas recientes, entregar obsequios de cuantías pequeñas para seguir fidelizando.
- Un margen del 44% es saludable para el sector comercial. El siguiente paso sería analizar rentabilidad detallada de productos por categoría y por canal para identificar prioridades de qué impulsar comercialmente.